# Credit card fraud detection — leakage-safe, reproducible workflow

This revised notebook fixes two methodological leaks from the earlier version:

1. **Split raw data before fitting StandardScaler.** Preprocessing now lives in a pipeline.
2. **Fit SMOTE only in the training portion of each CV fold.** Never resample globally before cross-validation.

The test set is never used for model selection or parameter tuning, and is evaluated **once** after the decision. Model selection uses **average precision (PR-AUC)** and F1 because fraud is extremely rare. No real dataset or historical performance metric is bundled.

**Run:** Download the [Kaggle Credit Card Fraud Detection data](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), put `creditcard.csv` in the current notebook working directory (Colab: upload or mount Drive), and run cells in order. The full dataset can take considerable CPU time, especially for grid search.

> Results in earlier README versions and checked-in PNG charts are **historical, not independently validated after fixing leakage**. Do not claim their numbers as results of this version.

In [ ]:
%pip -q install numpy pandas scikit-learn imbalanced-learn xgboost matplotlib seaborn joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
)
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
RANDOM_STATE = 42
print("Imports ready")

## 1 — Load Kaggle data (not committed to GitHub)

In Colab, you may instead mount your Drive and set `DATA_PATH` to an existing file path. Avoid uploading raw transaction data to public repositories.

In [ ]:
DATA_PATH = Path("creditcard.csv")  # Change to your Drive path if appropriate.
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Download Kaggle creditcard.csv; upload it or mount Google Drive."
    )
df = pd.read_csv(DATA_PATH)
assert "Class" in df and set(df["Class"].unique()).issubset({0, 1})
print("Rows:", len(df), "Fraud prevalence:", f"{df['Class'].mean():.3%}")

## 2 — Reserve untouched stratified test set

Neither feature scaling nor SMOTE runs before the split. **Do not inspect test-set metrics while selecting models.**

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
assert set(X_train.index).isdisjoint(set(X_test.index))
print(f"Training: {len(X_train):,} | sealed test: {len(X_test):,}")
print("Training fraud count:", int(y_train.sum()))

## 3 — Pipeline fits scaler and SMOTE inside each training fold

`sampling_strategy=0.10` is a computationally manageable experiment setting, **not** a validated optimal ratio. A balanced synthetic 1:1 set would be much larger. No SMOTE samples appear in validation or test folds.

In [ ]:
def make_pipeline(estimator):
    return Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(sampling_strategy=0.10, random_state=RANDOM_STATE)),
        ("model", clone(estimator)),
    ])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=1200, random_state=RANDOM_STATE)
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(n_estimators=80, n_jobs=2, random_state=RANDOM_STATE)
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(n_estimators=120, max_depth=5, n_jobs=2,
                      tree_method="hist", eval_metric="logloss",
                      random_state=RANDOM_STATE)
    ),
}
print("Candidates:", list(models))

## 4 — Compare models with *training-only* cross-validation

All metrics shown here are CV scores from the training partition, not the sealed test. Average precision (area under the precision-recall curve) is the selection criterion.

In [ ]:
comparison = []
for name, pipeline in models.items():
    print("Cross-validating:", name)
    s = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring={"ap": "average_precision", "f1": "f1"},
        n_jobs=1, error_score="raise"
    )
    comparison.append({
        "model": name,
        "CV mean PR-AUC": s["test_ap"].mean(),
        "CV sd PR-AUC": s["test_ap"].std(),
        "CV mean F1": s["test_f1"].mean(),
    })
comparison_df = pd.DataFrame(comparison).sort_values(
    "CV mean PR-AUC", ascending=False
)
display(comparison_df)
best_base_name = comparison_df.iloc[0]["model"]
best_base_score = float(comparison_df.iloc[0]["CV mean PR-AUC"])

## 5 — Tune Random Forest without leakage

GridSearchCV receives **raw X_train** and the full pipeline. It refits the scaler and SMOTE separately per fold. Its best CV result may have mild hyperparameter-selection optimism; the untouched holdout is the final evaluation.

In [ ]:
rf_grid = GridSearchCV(
    estimator=models["Random Forest"],
    param_grid={
        "model__max_depth": [12, None],
        "model__min_samples_leaf": [1, 3],
    },
    scoring="average_precision",
    cv=cv, n_jobs=1, refit=True, verbose=1,
)
rf_grid.fit(X_train, y_train)
print("RF best parameters:", rf_grid.best_params_)
print("RF grid CV average precision:", round(rf_grid.best_score_, 4))

## 6 — Final model choice (training data only) and one test evaluation

Choose via training-CV scores, fit on the full training partition, then calculate test metrics a single time. This is the **first time we predict on X_test**.

In [ ]:
if rf_grid.best_score_ > best_base_score:
    winner_name = "Tuned Random Forest"
    final_pipeline = rf_grid.best_estimator_  # Refit by GridSearchCV on training only.
else:
    winner_name = best_base_name
    final_pipeline = clone(models[best_base_name]).fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
y_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_results = {
    "Winner selected on training CV": winner_name,
    "Test sample size": len(y_test),
    "Test fraud cases": int(y_test.sum()),
    "Average precision (PR-AUC)": average_precision_score(y_test, y_prob),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "Accuracy (misleading for rare fraud)": accuracy_score(y_test, y_pred),
}
print(pd.Series(test_results))
print("Reminder: numbers will differ from historical charts and README scores.")

## 7 — Diagnostics of the final model

Charts are generated from the one held-out test evaluation. Avoid tuning or reselecting based on these visuals.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set(title=f"Confusion matrix: {winner_name}", xlabel="Predicted", ylabel="Actual")
p, r, _ = precision_recall_curve(y_test, y_prob)
axes[1].plot(r, p, label=f"AP = {average_precision_score(y_test, y_prob):.3f}")
axes[1].set(title="Precision-recall", xlabel="Recall", ylabel="Precision")
axes[1].legend()
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[2].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}")
axes[2].plot([0, 1], [0, 1], "--", color="gray")
axes[2].set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate")
axes[2].legend()
plt.tight_layout()
plt.savefig("leakage_safe_diagnostics.png", dpi=150)
plt.show()

## 8 — Feature importance of the selected estimator

Tree-based importances and absolute logistic coefficients are **model-dependent diagnostics**, not causal explanations.

In [ ]:
estimator = final_pipeline.named_steps["model"]
if hasattr(estimator, "feature_importances_"):
    importance = np.asarray(estimator.feature_importances_)
elif hasattr(estimator, "coef_"):
    importance = np.abs(np.asarray(estimator.coef_)[0])
else:
    importance = None
if importance is not None:
    top = pd.Series(importance, index=X_train.columns).sort_values(ascending=False).head(15)
    display(top.rename("importance").to_frame())
    top.sort_values().plot(kind="barh", figsize=(8, 5), title="Model-specific feature importance")
    plt.tight_layout()
    plt.show()
else:
    print("No native feature importances; consider held-out permutation importance.")

## 9 — Save the **entire fitted pipeline**, not a separately fitted scaler

Load only `joblib` files from trusted sources. The pipeline receives raw features on inference; it applies its fitted scaler and skips the training-only SMOTE sampler when predicting.

In [ ]:
bundle = {
    "pipeline": final_pipeline,
    "features": list(X_train.columns),
    "winner": winner_name,
    "metric": "test_average_precision",
    "test_average_precision": float(average_precision_score(y_test, y_prob)),
}
joblib.dump(bundle, "fraud_model_leakage_safe.joblib")
reloaded = joblib.load("fraud_model_leakage_safe.joblib")
example = X_test.iloc[:5][reloaded["features"]]
predictions = reloaded["pipeline"].predict(example)
probabilities = reloaded["pipeline"].predict_proba(example)[:, 1]
display(pd.DataFrame({"prediction": predictions, "fraud_probability": probabilities}, index=example.index))

## Limitations and next steps

- This is an **educational proof of methodology**, not a production fraud-detection system.
- No figures were recalculated for this revision without running the original Kaggle dataset; previous repository PNGs are historical.
- Test metrics are a one-time check, not repeated evidence of generalization. A temporal/out-of-period evaluation is recommended before practical claims.
- The default 0.5 classification threshold is not optimized for business costs; any threshold tuning must use **training/validation only**, not the held-out test.
- SHAP is omitted from this first correctness-focused baseline. Add it only after validating the chosen model, with fitting preprocessing applied consistently.
- Do not deploy unreviewed joblib artifacts or expose raw financial records.